## Introduction
Cancer treatment is not equally effective for every patient. Tumours that look similar clinically can respond very differently to the same anticancer drug. One reason for this is the variety of genomic alterations present in the cells. Some of these alterations affect the biological pathways a drug targets and/or the mechanisms by which a cancer cell can survive drug treatment. If we can characterize the molecular features of a cancer cell, perhaps we can predit which drugs are likely to work on that cell. The Genomics of Drug Sensitivity in Cancer (GDSC) project has generated a large dataset combining:

- cancer cell lines
- genomic/molecular characteristics of those cell lines
- measurement of their responses to anticancer compounds

This makes the GDSC well suited to investigating the relationship between genomic features and drug response. 

Because the genomic feature space is enormous, Machine Learning (ML) provides methods for efficiently learning the relationships between genomic features and drug response. Importantly, it is uncertain whether genomic information actually contains enough signal to predict drug response.

**Question:** Can anticancer drug response be predicted from genomic features using machine-learning models trained on GDSC data, and which genomic features contribute most to predictive performance?

This question can be broken down into sevearal sub-questions:
1. **Predictability:** Is anticancer drug response predictable from genomic features?
2. **Model Performance:** Which machine-learning models provice the most useful predictive performance? Which models are most efficient? Is there an intersection between performance and efficiency?
3. **Generalization:** Does the predictive performance of the models generalize to unseen data?
4. **Biological Interpretation:** Which genomic features contribute most to predictive performance? Can we interpret the models to understand the biological mechanisms underlying drug response?
5. **Drug Specificity:** Are there specific drugs for which genomic features are particularly predictive of response? Conversely, are there drugs for which genomic features provide little predictive power?

**Null Hypothesis:** Genomic features do not contain enough information to predict anticancer drug response, with the methods explored.

**Alternative Hypothesis:** Genomic features exist that can provide predictive information about anticancer drug response, and the machine-learning models exploured can be trained to leverage this information effectively.

## Data Ingestion
### Overview

The data ingestion phase established a reproducible pipeline for acquiring and combining the datasets required to investigate whether anticancer drug response can be predicted from genomic features.

The analysis uses GDSC release 8.4, released July 24, 2022, for drug-response measurements and cell-line metadata. Gene-expression features are obtained separately from the COSMIC Cell Lines Project, version 104, using the GRCh38 release. The two sources use different identifiers, so an explicit mapping procedure was required before the datasets could be joined.

The goal of this phase was to acquire the original source data without modifying it, validate its structure, and construct a reproducible representation suitable for subsequent preprocessing.

### GDSC response data

The primary drug-response data were obtained from the official Sanger CancerRxGene GDSC release 8.4. The release contains two fitted single-agent response datasets:

- GDSC1 fitted dose-response data
- GDSC2 fitted dose-response data

Both datasets were downloaded and retained rather than selecting one at the ingestion stage. They were concatenated into a single response table while retaining the DATASET field so that the origin of each observation remains identifiable.

The response data contain, among other fields:

- `COSMIC_ID`
- `CELL_LINE_NAME`
- `DRUG_NAME`
- `AUC`
- `LN_IC50`
- `DATASET`

Both AUC and LN_IC50 were retained at this stage. Selection of the final response variable is a modelling decision and therefore belongs to the preprocessing/analysis phase rather than data acquisition.

### GDSC cell-line metadata

The GDSC release also provides Cell_Lines_Details.xlsx. This file was downloaded alongside the response data and used to attach biological metadata to the response observations.

The relevant metadata include:

- GDSC tissue of origin
- secondary tissue descriptor
- TCGA cancer type
- COSMIC identifier
- sample name
- availability indicators for WES, CNA, gene expression, and methylation

The tissue-of-origin field is particularly important because the research question is intended to support cancer-location-specific analyses rather than treating all cancer cell lines as a single homogeneous population.

The metadata are joined to the response data using the numeric COSMIC_ID supplied by GDSC.

### Gene-expression data

The GDSC release metadata indicate whether gene-expression data are available for each cell line, but the actual genomic feature matrix is not contained in the GDSC response files. Consequently, a second data source was required.

The selected genomic modality is gene expression from the COSMIC Cell Lines Project.

The downloaded COSMIC product is:

- Cell Lines Project Complete Gene Expression
- Version 104
- GRCh38
- Affymetrix Human Genome U219 Array

The data are supplied as a compressed TSV file contained within a TAR archive:

`CellLinesProject_CompleteGeneExpression_v104_GRCh38.tar`

containing:

`CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz`

The expression file is initially in long format, with each row representing a cell-line/gene combination. The relevant fields are:

- COSMIC_SAMPLE_ID
- SAMPLE_NAME
- COSMIC_GENE_ID
- GENE_SYMBOL
- REGULATION
- Z_SCORE
- COSMIC_STUDY_ID

For this project, Z_SCORE is used as the expression measurement.

## Handling the COSMIC download

Unlike the historical GDSC release files, the current COSMIC download does not provide a permanent public URL. COSMIC generates a user-specific, time-limited signed URL.

Because the URL contains credentials and an expiration timestamp, it should not be committed to the repository. Instead, the current URL is supplied through an environment variable:

`COSMIC_LINK`

The ingestion code loads this value from `.env` using `python-dotenv`.

This approach allows the data-acquisition code to remain reproducible without embedding a user`s private, temporary COSMIC download URL in source control.

A complication encountered during development was that the signed URL could return HTTP 403 errors when it had expired. Refreshing the COSMIC download URL and placing the new value in .env resolved the problem. The already-downloaded archive is subsequently reused, so the signed URL is not required every time the dataset is loaded.

## Archive extraction

The COSMIC download is distributed as a TAR archive rather than directly as the expression TSV. The ingestion code therefore:

1. Obtains the signed URL from COSMIC_LINK.
2. Downloads the TAR archive to data/raw.
3. Inspects the archive contents.
4. Locates the expected expression file.
5. Extracts CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz.
6. Retains the original compressed expression file for subsequent loading.

The archive and extracted expression file are treated as raw inputs rather than modified analytical datasets.

## Mapping GDSC to COSMIC

A major challenge was that GDSC and COSMIC do not use the same identifier for the cell lines.

GDSC provides a numeric `COSMIC_ID`, whereas the COSMIC expression dataset identifies samples using `COSMIC_SAMPLE_ID`, such as `COSS905985`.

The COSMIC sample file establishes the relationship between the COSMIC sample identifier and `SAMPLE_NAME`. The GDSC cell-line metadata also contains `Sample Name`.

Therefore, the mapping was established through the cell-line sample name:

`GDSC COSMIC_ID → GDSC Sample Name → COSMIC SAMPLE_NAME → COSMIC_SAMPLE_ID`

This was preferable to attempting to infer the relationship from the cell-line names in the drug-response table alone.

An empirical validation of this mapping was performed before incorporating expression data.

The comparison found:

- 1,002 GDSC sample names
- 1,020 COSMIC sample names
- 999 matching names
- 99.7% of GDSC names matched to COSMIC
- no COSMIC sample names mapped to multiple COSMIC sample IDs

Three GDSC names did not have a corresponding COSMIC sample name:

- `GT3TKB`
- `Hep 3B2_1-7`
- `TOTAL:`

`TOTAL:` was identified as a summary row rather than a biological cell line and was explicitly excluded from the mapping process.

The remaining unmatched names are retained as unmatched rather than being assigned an inferred identity.

## Expression data duplication

During integration, an apparent uniqueness problem was discovered in the COSMIC expression data.

The raw expression dataset contains approximately 17.5 million rows. Initial inspection showed approximately 1.86 million duplicate rows, indicating that the assumption that every `COSMIC_SAMPLE_ID`/`GENE_SYMBOL` combination was unique was incorrect.

Further investigation showed that the duplication was associated primarily with two COSMIC studies:

- COSU3
- COSU619

The duplication was not simply a matter of identical repeated records. Within `COSU619`, 11,688 sample/gene pairs occurred twice. Of those pairs, 9,814 had identical Z-scores, while 1,874 had slightly different Z-scores.

The two genes involved were:

- `ATXN7`
- `TMSB15B`

The differences were extremely small. For example, the largest observed differences were approximately 0.013 Z-score units.

This investigation was important because it demonstrated that blindly using `pivot()` with an assumed unique sample/gene key would be inappropriate. It also established that the duplicate records need to be understood as part of the COSMIC data structure rather than treated automatically as errors.

The ingestion phase therefore does not silently discard these observations. The duplicate structure is documented and must be resolved explicitly during preprocessing before constructing the final feature matrix.

## Long-to-wide transformation

The COSMIC expression source is distributed in long format:

`sample × gene → expression value`

Machine-learning models require a feature matrix in which each observation has a set of genomic features. The eventual representation therefore needs to be transformed into:

`cell line → gene 1, gene 2, gene 3, ...`

with the corresponding Z-score for each gene.

The ingestion code contains a preparation step for this transformation, but the duplicate observations discovered above mean that the final aggregation rule should be established during preprocessing rather than hidden inside the raw data-ingestion step.

This separation is intentional: ingestion should preserve the source data and expose structural problems, while preprocessing should document and apply the analytical decisions used to resolve them.

## Reproducibility and validation

The ingestion pipeline records the GDSC release information and the COSMIC expression product used. A manifest is written to the raw-data directory containing:

- GDSC release number
- GDSC release date
- source location
- source filenames
- COSMIC expression product
- COSMIC version
- genome build
- expected archive and expression filenames

Validation functions check that the expected GDSC response files, cell-line metadata, and COSMIC expression file exist and contain the required columns.

The pipeline is designed so that existing downloaded files are reused rather than downloaded repeatedly. This is particularly important for the COSMIC dataset because its download URL is temporary and user-specific.

## Result of the ingestion phase

At the end of data ingestion, the project has three principal raw data resources:

1. GDSC1 drug-response data
2. GDSC2 drug-response data
3. GDSC cell-line metadata
4. COSMIC v104 gene-expression data

The response data and metadata can be joined using `COSMIC_ID`. The expression data can be linked to the GDSC cell lines through the validated `SAMPLE_NAME` mapping.

No biological observations have yet been removed because of missing genomic features, no expression values have been imputed, no genes have been selected or filtered, and no machine-learning train/test split has been performed.

These decisions are deliberately deferred to the preprocessing stage.

The key output of ingestion is therefore *a validated, reproducible connection between drug-response observations, cancer/tissue metadata, and COSMIC gene-expression measurements*, while preserving enough information about the original sources and identifier mappings to make the subsequent analysis auditable.


We are investigating the merge of the GDSC set with the COSMIC set, it isn't as straightforward as it initially seemed.

### Initial dataset characterization

The following summaries describe the loaded release before any preprocessing or analytical filtering. Tissue and cancer labels are retained as metadata; no cancer type is selected at this stage.

### Genomic availability and quality checks

The response files contain no genomic feature matrix. `Cell_Lines_Details.xlsx` records whether WES, CNA, gene expression, and methylation assays are available for each model; the actual feature files and their identifiers must be selected and joined in a later research-design step. The checks below report raw-data limitations without removing observations.

### Data-to-Preprocessing checkpoint

This analysis uses official Sanger CancerRxGene GDSC release 8.4 (24 July 2022), combining GDSC1 and GDSC2 fitted single-agent response files with `Cell_Lines_Details.xlsx`. Each observation represents a cell-line/drug response record and retains COSMIC model ID, tissue of origin, and cancer metadata. AUC and LN_IC50 are available and complete in the loaded release; neither is selected as the final target yet.

The release metadata reports WES, CNA, gene-expression, and methylation availability flags, but does not include the genomic feature matrices themselves. Cancer type is missing for a substantial subset of observations, while tissue labels are retained. Before preprocessing, the project still needs to choose a genomic modality and source, define eligibility and missingness rules, select a response measure, and decide how cancer types will be compared. No observations have been imputed, filtered, normalized, or split at this checkpoint.

### Reproducible preprocessing interface

The preprocessing implementation lives in `gdsc.preprocessing`, so the notebook does not duplicate hidden cleaning steps. It requires a selected genomic matrix with one verified shared cell-line identifier (normally `COSMIC_ID`). The downloaded release inspected above does not supply that matrix: its availability flags only indicate whether an assay exists for a model. Therefore executing a real join here would falsely imply that a genomic modality has been selected.

Once a feature matrix is obtained, call `preprocess_gdsc` with the documented study thresholds. The returned `X` will contain genomic columns only; `y` will be the explicit AUC or LN_IC50 choice; and tissue, cancer type, drug, and identifiers will remain in `metadata`. The function records every row/feature count in `prepared.info`. It does not fit an imputer or scaler. Any later transformer must be fit on the training split only.

The next cell is deliberately a readiness check, not a call to `preprocess_gdsc`: no genomic feature matrix is currently available. This checkpoint intentionally stops before a train/test split and model training.

### Reproducible preprocessing interface

The preprocessing implementation lives in `gdsc.preprocessing`, so the notebook does not duplicate hidden cleaning steps. It requires a selected genomic matrix with one verified shared cell-line identifier (normally `COSMIC_ID`). The downloaded release inspected above does not supply that matrix: its availability flags only indicate whether an assay exists for a model. Therefore executing a real join here would falsely imply that a genomic modality has been selected.

Once a feature matrix is obtained, call `preprocess_gdsc` with the documented study thresholds. The returned `X` will contain genomic columns only; `y` will be the explicit AUC or LN_IC50 choice; and tissue, cancer type, drug, and identifiers will remain in `metadata`. The function records every row/feature count in `prepared.info`. It does not fit an imputer or scaler. Any later transformer must be fit on the training split only.

No preprocessing call is shown here because no genomic feature matrix is currently available. This checkpoint intentionally stops before a train/test split and model training.

### Memory-safe expression access

A previous ingestion attempt merged the complete COSMIC expression matrix (about 17,000 genes) onto all GDSC drug-response observations (about 575,000 rows). That would create a dense response-row-by-gene table, and it failed with a memory allocation request of roughly 72.8 GiB. This was an architecture problem, not a reason to discard observations or genes.

The replacement is a feature-store workflow. GDSC response records and metadata remain a lightweight table. COSMIC expression is de-duplicated using the documented arithmetic mean for repeated sample/gene Z-scores and cached once as long-format Parquet in `data/processed/cosmic_expression.parquet`. During preprocessing, the project will first select the cohort, cell lines, and genes, then call `load_expression_features` for only that subset. Only that bounded subset may be merged into a modelling dataset.

This preserves the raw files, avoids repeating the expensive preparation step, and prevents a future full-expression/full-response merge from recreating the memory failure.

### Analysis pipeline

```text
Raw source data
│
├── GDSC1 / GDSC2 drug-response files
├── GDSC cell-line metadata
└── COSMIC v104 gene-expression data
        │
        ▼
Data ingestion and preparation
│
├── Validate source files and schemas
├── Standardize identifiers and metadata
├── Map GDSC cell lines to COSMIC samples
├── Resolve documented duplicate expression records
└── Cache expression data as Parquet
        │
        ▼
Analytical source data
│
├── GDSC response + metadata table
└── COSMIC expression feature store
        │
        ▼
Preprocessing
│
├── Select tissue / cancer cohort
├── Select response variable
├── Define drug eligibility
├── Define cell-line eligibility
├── Query relevant expression features
├── Handle missing data
├── Filter / select genes
├── Construct feature matrix X and target y
├── Split training / validation / test data
└── Apply scaling or transformations as appropriate
        │
        ▼
Modeling
│
├── Train baseline models
├── Train candidate machine-learning models
├── Tune model hyperparameters
└── Evaluate predictive performance
        │
        ▼
Interpretation
│
├── Compare model performance
├── Estimate feature importance
├── Identify genomic features associated with prediction
└── Assess biological and methodological implications


The important boundary is that the **Parquet feature store is the output of ingestion/preparation, not the output of ML preprocessing**.

### Preprocessing decisions implemented in the module

Preprocessing now uses a drug-specific observation unit: one response per eligible cell line for one selected drug. The module first selects a tissue, summarizes drug eligibility within that cohort, maps only the retained cell lines to COSMIC, and requests only the chosen genes from the Parquet store. This prevents the former all-responses by all-genes memory failure from returning.

The resulting `X` contains expression Z-scores only; `y` is an explicit AUC or LN_IC50 choice; cell-line identifiers, drug, tissue, cancer type, and dataset remain separate metadata. Missingness and variance are reported before filtering. Any imputation, variance filtering, or optional scaling is represented by an unfitted scikit-learn pipeline and must be fit on training data only. Splits are grouped by `COSMIC_ID`, preventing one cell line from leaking into more than one split.

In [18]:
from gdsc.data import load_gdsc

gdsc = load_gdsc(
    data_dir="../data/raw",
    include_metadata=True,
)

gdsc.head()

/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,COSMIC_ID,CELL_LINE_NAME,SANGER_MODEL_ID,TCGA_DESC,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,...,Copy Number Alterations (CNA),Gene Expression,Methylation,Drug\nResponse,TISSUE_OF_ORIGIN,TISSUE_DESCRIPTOR_2,CANCER_TYPE,Microsatellite \ninstability Status (MSI),Screen Medium,Growth Properties
0,GDSC1,361,17635802,684057,ES5,SIDM00263,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
1,GDSC1,361,17636176,684059,ES7,SIDM00269,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
2,GDSC1,361,17636568,684062,EW-11,SIDM00203,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
3,GDSC1,361,17636912,684072,SK-ES-1,SIDM01111,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Semi-Adherent
4,GDSC1,361,17637300,687448,COLO-829,SIDM00909,SKCM,1,Erlotinib,EGFR,...,Y,Y,Y,Y,skin,melanoma,SKCM,MSS/MSI-L,R,Adherent


## Preprocessing Question 1: what tissues give us enough cell lines and drug-response coverage to work with?

### Selecting a tissue cohort

Before constructing the machine-learning dataset, the analysis must define a tissue of origin. Because model performance depends on having enough independent cell lines with measured drug responses, tissue selection should be informed by the coverage available in GDSC rather than chosen arbitrarily.

The following summary compares each tissue by its number of unique cell lines, tested drugs, and total drug-response observations. In particular, the number of unique cell lines is important because cell lines represent the independent biological samples available for training and evaluating drug-specific models.

In [19]:
from gdsc.preprocessing import summarize_tissues

# Reusable summary; missing tissue labels are shown rather than silently removed.
tissue_summary = summarize_tissues(gdsc)
tissue_summary

,CELL_LINES,DRUGS,RESPONSE_OBSERVATIONS,cell_lines,observations
TISSUE_OF_ORIGIN,,,,,
lung_NSCLC,108,542,64499,108,64499
urogenital_system,104,542,61288,104,61288
leukemia,84,542,50008,84,50008
aero_dig_tract,77,542,45354,77,45354
lymphoma,69,542,40948,69,40948
lung_SCLC,63,542,33558,63,33558
skin,58,542,33253,58,33253
nervous_system,55,542,32794,55,32794
breast,52,542,31021,52,31021


### Tissue selection -> Lung

The tissue-coverage analysis identified lung_NSCLC as the largest available cohort, containing 108 unique cell lines, 542 tested drugs, and 64,499 drug-response observations. Because the subsequent analysis will construct drug-specific models, the number of unique cell lines is more important than the total number of response records: each selected drug will be represented by only the subset of cell lines in which it was tested.

For the initial analysis, lung_NSCLC will therefore be used as the tissue cohort. Selecting the largest available cohort maximizes the number of independent biological samples available for model development and evaluation while establishing a workflow that can subsequently be repeated for other tissues.

### Drug-response coverage within the NSCLC cohort

We now select `lung_NSCLC` using the reusable preprocessing function. Drug coverage is then summarized within this cohort only. `N_CELL_LINES`, not raw response rows, is the primary descriptive sample-size statistic because a future drug-specific model has one row per cell line. Duplicate drug/cell-line records are diagnosed below but are not averaged or removed at this stage.


In [20]:
import importlib
import gdsc.preprocessing as preprocessing

# Reload the local module so this cell reflects edits made during notebook development.
preprocessing = importlib.reload(preprocessing)

# The module performs all reusable selection and diagnostic calculations.
# This notebook only presents the documented results for the chosen cohort.
coverage_report = preprocessing.analyze_cohort_drug_coverage(
    gdsc,
    tissue_of_origin="lung_NSCLC",
    thresholds=(20, 30, 40, 50, 60, 75, 90, 100),
)
drug_summary = coverage_report["drug_summary"]
identity_diagnostics = coverage_report["identity_diagnostics"]
duplicate_diagnostics = coverage_report["duplicate_diagnostics"]
coverage_diagnostics = coverage_report["coverage_diagnostics"]
eligibility_decision_table = coverage_report["eligibility_decision_table"]

print(coverage_diagnostics["distribution"])
display(eligibility_decision_table)
display(drug_summary.head(20))
print({key: len(value) for key, value in identity_diagnostics.items()})
{key: value for key, value in duplicate_diagnostics.items() if key != "duplicated_pairs"}


count    542.000000
mean      95.201107
std       21.824840
min        5.000000
25%       98.250000
50%      104.000000
75%      108.000000
max      108.000000
Name: N_CELL_LINES, dtype: float64


,minimum_unique_cell_lines,eligible_drugs
0,20,537
1,30,510
2,40,510
3,50,508
4,60,501
5,75,501
6,90,411
7,100,381


,DRUG_NAME,N_OBSERVATIONS,N_CELL_LINES,N_AUC_AVAILABLE,AUC_MISSING_FRACTION,N_LN_IC50_AVAILABLE,LN_IC50_MISSING_FRACTION,DATASETS,PUTATIVE_TARGET,DRUG_ID,observations,cell_lines
DRUG_NAME,,,,,,,,,,,,
Selumetinib,Selumetinib,394,108,394,0.0,394,0.0,"(GDSC1, GDSC2)","(MEK1, MEK2,)","(1062, 1498, 1736)",394,108
AZD4547,AZD4547,322,108,322,0.0,322,0.0,"(GDSC1, GDSC2)","(FGFR1, FGFR2, FGFR3, FGRF1, FGFR2, FGFR3)","(1135, 1497, 1786)",322,108
PLX-4720,PLX-4720,321,108,321,0.0,321,0.0,"(GDSC1, GDSC2)","(BRAF,)","(1036, 1371)",321,108
AZD7762,AZD7762,320,108,320,0.0,320,0.0,"(GDSC1, GDSC2)","(CHEK1, CHEK2,)","(1022, 1402)",320,108
Olaparib,Olaparib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PARP1, PARP2,)","(1017, 1495)",319,108
Pictilisib,Pictilisib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PI3K (class 1),)","(1058, 1527)",319,108
Afatinib,Afatinib,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(EGFR, ERBB2,)","(1032, 1377)",318,108
SN-38,SN-38,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(TOP1,)","(1490, 1494)",318,108
Avagacestat,Avagacestat,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Amyloid beta20, Amyloid beta40,)","(205, 1072)",315,108


{'name_to_multiple_ids': 71, 'id_to_multiple_names': 0}


{'n_drug_cell_line_records': 64499,
 'n_unique_drug_cell_line_pairs': 57786,
 'n_duplicated_drug_cell_line_pairs': 6713,
 'max_records_per_pair': 2}

## Defining the first reproducible experiment

Drug-specific models need enough independent cell lines for development and evaluation. We therefore require at least 75 unique lung_NSCLC cell lines per drug. The initial candidate is selected strictly by coverage, then by lowest DRUG_ID in a tie. AUC is the primary continuous response: it summarizes the fitted dose-response curve, while LN_IC50 remains in the source data for a later sensitivity analysis.

GDSC1 and GDSC2 measurements are separate screens, not values to average. We choose the screen with the most usable AUC measurements for the selected drug (GDSC1 wins only an exact tie), then verify that every stable DRUG_ID/COSMIC_ID pair is unique within that source.

In [ ]:
experiment_response = preprocessing.build_initial_response_cohort(
    gdsc, tissue_of_origin="lung_NSCLC", min_unique_cell_lines=75, response_metric="AUC"
)
selected_drug = experiment_response["selected_drug"]
response_cohort = experiment_response["response_cohort"]
display(experiment_response["eligibility"]["eligible_drugs"].head(20))
display(selected_drug[["DRUG_ID", "DRUG_NAME", "N_CELL_LINES", "N_OBSERVATIONS", "DATASETS", "PUTATIVE_TARGET"]].to_frame().T)
display(experiment_response["dataset_coverage"])
print({
    "selected_dataset": experiment_response["selected_dataset"],
    "screen_specific_drug_id": experiment_response["selected_drug_id"],
    "response_rows": len(response_cohort),
    "unique_cell_lines": response_cohort["COSMIC_ID"].nunique(),
    "missing_auc_rows_excluded": experiment_response["n_excluded_response_rows"],
})


### Why the files have different roles

Raw GDSC CSV files, the metadata workbook, and compressed COSMIC TSV files in `data/raw/` are preserved source inputs. The TSV is too large to load as one dataframe, so cache construction reads it in chunks and uses a temporary SQLite database only to aggregate duplicate sample/gene Z-scores. That temporary database is deleted afterwards. The durable result is a long-format Parquet file in `data/processed/`; Parquet is used because it can retrieve only the selected COSMIC sample IDs. Only this selected response cohort is pivoted to a cell-line-by-gene matrix. No all-responses-by-all-genes table is created.

### Drug eligibility decision table

A drug is *eligible* at a stated threshold when it has response measurements from at least that many **unique cell lines**. The table is calculated by `filter_eligible_drugs`; raw response rows are not treated as independent biological samples. For the approved initial experiment, the threshold is **75 unique lung_NSCLC cell lines**. It remains a parameter so the workflow can be repeated under a different documented criterion.

The first experiment uses a deterministic availability rule rather than expected biology or predictive results: among eligible drugs, `select_initial_drug` chooses the greatest cell-line coverage and breaks an exact tie by the lowest `DRUG_ID`. AUC is the primary target; LN_IC50 remains available for later sensitivity analysis. GDSC1 and GDSC2 measurements are not averaged. Instead, `select_response_dataset` chooses the source screen with the greatest usable AUC coverage (GDSC1 only wins an exact tie) and raises an error if duplicate stable drug-ID/cell-line records remain within that screen.

## Preprocessing status

The data-ingestion phase is complete. GDSC release 8.4 drug-response data and cell-line metadata can be loaded successfully, and COSMIC gene-expression data have been integrated through a separate, memory-safe feature store. The ingestion pipeline handles GDSC/COSMIC sample mapping, duplicate expression measurements, caching, and targeted expression access without constructing an impractically large response-by-gene table.

The preprocessing phase has now begun. The first step was to examine the number of independent cell lines available for each tissue of origin. lung_NSCLC was selected for the initial analysis because it contains the largest cohort in the dataset, with 108 unique cell lines, 542 drugs, and 64,499 response observations. The workflow remains parameterized so that the same analysis can later be repeated for other tissues.

Drug eligibility is now parameterized and shown as a threshold decision table. Although 542 drugs are represented, not every drug was necessarily tested against all 108 cell lines. Because the eventual models will be drug-specific, the number of unique cell lines with a response measurement for each drug determines the effective sample size for that model.

The first reproducible experiment is now defined: 75 unique cell lines for eligibility, deterministic coverage-based drug selection, AUC as the response metric, and one GDSC screen selected by usable cell-line coverage. This resolves cross-screen duplication by choosing a consistent experimental source, never by averaging. Within-screen duplicate stable drug-ID/cell-line records remain a hard error.

The implemented path now constructs the resolved response cohort first, maps only those cell lines to COSMIC, and queries only those expression records from the Parquet feature store. The raw CSV/TSV/XLSX files remain source inputs in `data/raw/`. SQLite is used only as a temporary on-disk aggregation workspace while converting the large COSMIC TSV in chunks; it is deleted after the cache is built. The durable long-format Parquet cache is stored in `data/processed/` and is queried only after cohort selection. A bounded cell-line-by-gene matrix is then produced; the prohibited full response-row-by-gene merge is never created.

Current milestone: the approved response cohort, targeted expression integration, unsupervised missingness/variance filtering, and grouped train/validation/test handoff are implemented. Learned imputation and optional scaling remain unfitted until the training split, and no predictive model is fit in preprocessing.